# Subfase 8: Validación estadística (manifests)

Objetivo: preparar el entorno para comparar modelos clásicos y cuánticos sin reentrenar.

Salida (slice 2): CSVs con predicciones y pruebas estadísticas.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from bvg_core.config import TEST_SIZE
from bvg_core.splits import temporal_split_company
from bvg_core.quantum import build_qkernel
from bvg_core.utils import load_manifest, safe_company
from sklearn.metrics import accuracy_score, f1_score
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.proportion import proportion_confint

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)


In [2]:
ROOT = Path('..')
DATASET_PATH = ROOT / 'data' / 'processed' / 'BVG_features_svc_master.csv'
CLASSICAL_DIR = ROOT / 'models' / 'classical'
QUANTUM_DIR = ROOT / 'models' / 'quantum'
OUT_DIR = ROOT / 'results' / 'fase4_comparativa'
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_STATS_FILE = OUT_DIR / 'BVG_subfase8_test_estadistico.csv'
OUT_CONTINGENCY_FILE = OUT_DIR / 'BVG_subfase8_tablas_contingencia.csv'

DATE_COL = 'fecha'
COMPANY_COL = 'empresa'
TARGET_COL_DEFAULT = 'target_up_h5'

REQUIRED_MANIFEST_FIELDS = [
    'company',
    'model_family',
    'feature_columns',
    'target_column',
    'train_end_date',
]


In [3]:
# Manifest loading helpers removed — using load_manifest from bvg_core.utils directly.


In [4]:
classical_paths = sorted(CLASSICAL_DIR.glob('*_manifest.json'))
quantum_paths = sorted(QUANTUM_DIR.glob('*_manifest.json'))
expected_paths = classical_paths + quantum_paths

if not expected_paths:
    raise FileNotFoundError(
        f'No se encontraron manifests. Se esperaban archivos *_manifest.json en:'
        f'- {CLASSICAL_DIR.as_posix()}'
        f'- {QUANTUM_DIR.as_posix()}'
    )

manifests = []
for path in expected_paths:
    payload = load_manifest(path, required_fields=REQUIRED_MANIFEST_FIELDS)
    if not isinstance(payload.get('feature_columns'), list):
        raise ValueError(f'Manifest {path.name} tiene feature_columns inválido (se esperaba list).')
    manifests.append(payload)
if not manifests:
    raise ValueError('No se encontraron manifests para procesar.')
classical_manifests = [m for m in manifests if m.get('model_family') == 'classical']
quantum_manifests = [m for m in manifests if m.get('model_family') == 'quantum']

if not classical_manifests:
    raise ValueError('No se encontraron manifests clásicos para procesar.')
if not quantum_manifests:
    raise ValueError('No se encontraron manifests cuánticos para procesar.')


In [5]:
if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Dataset no encontrado: {DATASET_PATH}')

df = pd.read_csv(DATASET_PATH)

required_cols = {DATE_COL, COMPANY_COL, TARGET_COL_DEFAULT}
missing_cols = sorted(required_cols.difference(df.columns))
if missing_cols:
    raise ValueError(f'Columnas faltantes en dataset: {missing_cols}')

df = df.assign(**{DATE_COL: pd.to_datetime(df[DATE_COL], errors='coerce')})
if df[DATE_COL].isna().any():
    raise ValueError('Hay fechas inválidas en columna fecha.')


In [6]:
feature_cols_by_company = {}
target_col_by_company = {}

for payload in manifests:
    company = payload['company']
    feature_cols = payload['feature_columns']
    target_col = payload.get('target_column', TARGET_COL_DEFAULT)

    missing = [col for col in feature_cols + [target_col] if col not in df.columns]
    if missing:
        raise ValueError(
            f'Columnas faltantes en dataset para {company}: {missing}'
        )

    feature_cols_by_company[company] = feature_cols
    target_col_by_company[company] = target_col


In [7]:
splits_by_company = {}
clean_df_by_company = {}

for company in sorted(feature_cols_by_company.keys()):
    feature_cols = feature_cols_by_company[company]
    target_col = target_col_by_company[company]

    df_company = df.loc[df[COMPANY_COL] == company]
    df_company_clean = df_company.dropna(subset=[target_col])
    clean_df_by_company[company] = df_company_clean

    tr, te, X_train, y_train, X_test, y_test = temporal_split_company(
        df_company_clean,
        company,
        test_size=TEST_SIZE,
        company_col=COMPANY_COL,
        date_col=DATE_COL,
        target_col=target_col,
        feature_cols=feature_cols,
    )

    splits_by_company[company] = {
        'train_df': tr,
        'test_df': te,
        'X_train': X_train,
        'y_train': y_train,
        'X_test': X_test,
        'y_test': y_test,
    }


In [8]:
classical_artifacts = {}

for payload in classical_manifests:
    company = payload['company']
    tag = safe_company(company)
    pipeline_path = CLASSICAL_DIR / f'{tag}_h5_pipeline.joblib'
    if not pipeline_path.exists():
        raise FileNotFoundError(f'Pipeline clásico no encontrado: {pipeline_path}')

    classical_artifacts[company] = {
        'manifest': payload,
        'pipeline': joblib.load(pipeline_path),
        'pipeline_path': pipeline_path,
    }


In [9]:
quantum_artifacts = {}

for payload in quantum_manifests:
    company = payload['company']
    tag = safe_company(company)

    scaler_path = QUANTUM_DIR / f'{tag}_h5_scaler.joblib'
    pca_path = QUANTUM_DIR / f'{tag}_h5_pca.joblib'
    svc_path = QUANTUM_DIR / f'{tag}_h5_svc.joblib'
    kernel_config_path = QUANTUM_DIR / f'{tag}_h5_kernel_config.json'

    for path in [scaler_path, pca_path, svc_path, kernel_config_path]:
        if not path.exists():
            raise FileNotFoundError(f'Artifacto cuántico no encontrado: {path}')

    kernel_config = json.loads(kernel_config_path.read_text(encoding='utf-8'))
    qkernel = build_qkernel(kernel_config)

    feature_cols = feature_cols_by_company[company]
    train_end_date = payload.get('train_end_date')
    df_company_clean = clean_df_by_company.get(company)
    if df_company_clean is None:
        df_company = df.loc[df[COMPANY_COL] == company]
        df_company_clean = df_company.dropna(subset=[target_col_by_company[company]])

    scaler = joblib.load(scaler_path)
    pca = joblib.load(pca_path)
    svc = joblib.load(svc_path)

    # Usar el mismo train_df del split temporal para garantizar paridad
    tr = splits_by_company[company]['train_df']
    X_train_raw = tr.loc[:, feature_cols].copy()
    if X_train_raw.isna().any().any():
        raise ValueError(f"X_train tiene NaN para {company}")
    X_train_q = pca.transform(scaler.transform(X_train_raw))

    quantum_artifacts[company] = {
        'manifest': payload,
        'scaler': scaler,
        'pca': pca,
        'svc': svc,
        'kernel_config': kernel_config,
        'qkernel': qkernel,
        'X_train_q': X_train_q,
        'artifact_paths': {
            'scaler': scaler_path,
            'pca': pca_path,
            'svc': svc_path,
            'kernel_config': kernel_config_path,
        },
    }


In [10]:
def _get_proba(model, X):
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X)[:, 1]
    return np.full((X.shape[0],), np.nan)

def compute_contingency(y_true, y_pred_classical, y_pred_quantum):
    correct_classical = y_pred_classical == y_true
    correct_quantum = y_pred_quantum == y_true
    both_correct = int(np.sum(correct_classical & correct_quantum))
    classical_only = int(np.sum(correct_classical & ~correct_quantum))
    quantum_only = int(np.sum(~correct_classical & correct_quantum))
    both_wrong = int(np.sum(~correct_classical & ~correct_quantum))
    return both_correct, classical_only, quantum_only, both_wrong

def compute_mcnemar(b, c):
    use_correction = (b + c) < 25
    result = mcnemar([[0, b], [c, 0]], exact=False, correction=use_correction)
    return float(result.statistic), float(result.pvalue), use_correction

def compute_accuracy_ci(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    lower, upper = proportion_confint(
        count=int(np.sum(y_true == y_pred)),
        nobs=len(y_true),
        alpha=0.05,
        method='wilson',
    )
    return float(acc), float(lower), float(upper)


In [11]:
prediction_rows = []
stats_rows = []
contingency_rows = []

for company in sorted(splits_by_company.keys()):
    split = splits_by_company[company]
    X_test = split['X_test']
    y_test = split['y_test']
    test_df = split['test_df'].copy()

    if X_test.isna().any().any():
        raise ValueError(f'NaN detectado en X_test para {company}.')

    classical_bundle = classical_artifacts.get(company)
    quantum_bundle = quantum_artifacts.get(company)
    if not classical_bundle or not quantum_bundle:
        raise ValueError(f'Faltan artefactos para {company}.')

    pipeline = classical_bundle['pipeline']
    y_pred_classical = pipeline.predict(X_test)
    proba_classical = _get_proba(pipeline, X_test)

    scaler = quantum_bundle['scaler']
    pca = quantum_bundle['pca']
    svc = quantum_bundle['svc']
    qkernel = quantum_bundle['qkernel']
    X_train_q = quantum_bundle['X_train_q']
    X_test_q = pca.transform(scaler.transform(X_test))
    K_test = qkernel.evaluate(x_vec=X_test_q, y_vec=X_train_q)
    y_pred_quantum = svc.predict(K_test)
    proba_quantum = _get_proba(svc, K_test)

    for idx in range(len(test_df)):
        prediction_rows.append({
            'company': company,
            'fecha': test_df.iloc[idx][DATE_COL],
            'y_test': int(y_test.iloc[idx]),
            'y_pred_classical': int(y_pred_classical[idx]),
            'y_pred_quantum': int(y_pred_quantum[idx]),
            'proba_classical': float(proba_classical[idx]) if not np.isnan(proba_classical[idx]) else np.nan,
            'proba_quantum': float(proba_quantum[idx]) if not np.isnan(proba_quantum[idx]) else np.nan,
        })

    both_correct, classical_only, quantum_only, both_wrong = compute_contingency(
        y_test.to_numpy(),
        y_pred_classical,
        y_pred_quantum,
    )
    mcnemar_stat, mcnemar_pvalue, used_correction = compute_mcnemar(classical_only, quantum_only)

    acc, acc_low, acc_up = compute_accuracy_ci(y_test.to_numpy(), y_pred_classical)
    f1 = f1_score(y_test.to_numpy(), y_pred_classical, zero_division=0)

    stats_rows.append({
        'company': company,
        'model_family': 'classical',
        'n_test': int(len(y_test)),
        'accuracy': acc,
        'acc_ci_lower': acc_low,
        'acc_ci_upper': acc_up,
        'f1': float(f1),
        'mcnemar_chi2': mcnemar_stat,
        'mcnemar_pvalue': mcnemar_pvalue,
        'mcnemar_significativo': bool(mcnemar_pvalue < 0.05),
        'mcnemar_conclusion': 'Diferencia significativa' if mcnemar_pvalue < 0.05 else 'No significativa',
    })

    acc_q, acc_q_low, acc_q_up = compute_accuracy_ci(y_test.to_numpy(), y_pred_quantum)
    f1_q = f1_score(y_test.to_numpy(), y_pred_quantum, zero_division=0)

    stats_rows.append({
        'company': company,
        'model_family': 'quantum',
        'n_test': int(len(y_test)),
        'accuracy': acc_q,
        'acc_ci_lower': acc_q_low,
        'acc_ci_upper': acc_q_up,
        'f1': float(f1_q),
        'mcnemar_chi2': mcnemar_stat,
        'mcnemar_pvalue': mcnemar_pvalue,
        'mcnemar_significativo': bool(mcnemar_pvalue < 0.05),
        'mcnemar_conclusion': 'Diferencia significativa' if mcnemar_pvalue < 0.05 else 'No significativa',
    })

    contingency_rows.append({
        'company': company,
        'both_correct': both_correct,
        'classical_only': classical_only,
        'quantum_only': quantum_only,
        'both_wrong': both_wrong,
    })

    if used_correction:
        print(f'McNemar: Edwards correction aplicada para {company} (b+c < 25).')

pred_df = pd.DataFrame(prediction_rows)
stats_df = pd.DataFrame(stats_rows)
contingency_df = pd.DataFrame(contingency_rows)

if pred_df.empty:
    raise ValueError('No se generaron predicciones para exportar.')
if stats_df.empty:
    raise ValueError('No se generaron estadísticas para exportar.')
if contingency_df.empty:
    raise ValueError('No se generaron tablas de contingencia para exportar.')

stats_df = stats_df.sort_values(['company', 'model_family']).reset_index(drop=True)
contingency_df = contingency_df.sort_values(['company']).reset_index(drop=True)

stats_df.to_csv(OUT_STATS_FILE, index=False)
contingency_df.to_csv(OUT_CONTINGENCY_FILE, index=False)

print(f'CSV estadístico exportado en: {OUT_STATS_FILE.as_posix()}')
print(f'CSV contingencia exportado en: {OUT_CONTINGENCY_FILE.as_posix()}')


McNemar: Edwards correction aplicada para BANCO GUAYAQUIL S.A. (b+c < 25).
McNemar: Edwards correction aplicada para CORPORACION FAVORITA C.A. (b+c < 25).
CSV estadístico exportado en: ../results/fase4_comparativa/BVG_subfase8_test_estadistico.csv
CSV contingencia exportado en: ../results/fase4_comparativa/BVG_subfase8_tablas_contingencia.csv
